In [1]:
!pip install -q transformers timm einops

In [2]:
try:
    import transformers
    import timm
    import einops
    print(f"Transformers version: {transformers.__version__}")
    print(f"Timm version: {timm.__version__}")
    print("✅ Success! All libraries installed correctly. You can ignore the pip error.")
except ImportError as e:
    print(f"❌ Installation failed: {e}")

Transformers version: 4.53.3
Timm version: 1.0.19
✅ Success! All libraries installed correctly. You can ignore the pip error.


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoModelForImageClassification # Added for Mamba
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, f1_score, recall_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
import os
import glob
import random
from itertools import cycle

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [4]:

# ---------------------------------------------------------
# 1. Transforms (Updated to 224x224 for MambaVision)
# ---------------------------------------------------------
# MambaVision is optimized for 224x224. 
# We use standard ImageNet mean/std which matches the model config.
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])



In [5]:
# ---------------------------------------------------------
# 2. Dataset Class
# ---------------------------------------------------------
class UnifiedEmotionDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform
        
        # Determine classes dynamically from the list
        unique_labels = sorted(list(set([item[1] for item in self.file_list])))
        self.classes = unique_labels
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(unique_labels)}

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path, label_name = self.file_list[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        label = self.class_to_idx[label_name]
        return image, label

In [6]:
# ---------------------------------------------------------
# 3. Build the Master File List
# ---------------------------------------------------------
all_data = []

# --- Part A: Original Dataset (Excluding Ahegao) ---
root_dir_1 = "/kaggle/input/emotion-recognition-dataset/dataset"
folders_to_keep = ['Angry', 'Happy', 'Neutral', 'Sad', 'Surprise']

if os.path.exists(root_dir_1):
    for class_name in folders_to_keep:
        folder_path = os.path.join(root_dir_1, class_name)
        images = glob.glob(os.path.join(folder_path, "*")) 
        for img_path in images:
            all_data.append((img_path, class_name))
else:
    print(f"Warning: Directory {root_dir_1} not found. Ensure dataset path is correct.")

# --- Part B: AffectNet (Adding Disgust & Fear) ---
def get_affectnet_files(folder_path, label, oversample=False, target_count=6000):
    valid_extensions = (".jpg", ".png", ".jpeg", ".tiff")
    files = []
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            if any(filename.lower().endswith(ext) for ext in valid_extensions):
                files.append((os.path.join(folder_path, filename), label))
    else:
        print(f"Warning: Path {folder_path} does not exist.")
        return []
            
    if oversample and len(files) > 0:
        oversample_factor = max(1, target_count // len(files))
        final_files = []
        for _ in range(oversample_factor):
            final_files.extend(files)
        # Fill remainder
        final_files.extend(files[:target_count % len(files)])
        return final_files
    return files

# Paths
disgust_path = "/kaggle/input/affectnet/archive (3)/Train/disgust"
fear_path = "/kaggle/input/affectnet/archive (3)/Train/fear"

# Add Fear
all_data.extend(get_affectnet_files(fear_path, 'Fear', oversample=False))
# Add Disgust (Oversampled)
all_data.extend(get_affectnet_files(disgust_path, 'Disgust', oversample=True, target_count=6000))



In [7]:
# ---------------------------------------------------------
# 4. Manual Split & Dataset Creation
# ---------------------------------------------------------

# 1. Shuffle the raw list of strings
random.shuffle(all_data)

# 2. Calculate indices
total_len = len(all_data)
train_end = int(0.8 * total_len)
val_end = int(0.9 * total_len) 

# 3. Slice the lists
train_files = all_data[:train_end]
val_files   = all_data[train_end:val_end]
test_files  = all_data[val_end:]

# 4. Create Datasets with CORRECT Transforms
train_dataset = UnifiedEmotionDataset(train_files, transform=train_transforms)
val_dataset   = UnifiedEmotionDataset(val_files, transform=test_transforms)
test_dataset  = UnifiedEmotionDataset(test_files, transform=test_transforms)

# 5. DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 6. Verification
class_names = train_dataset.classes # Save for plotting later
NUM_CLASSES = len(class_names)
print("Classes detected:", class_names)
print(f"Total Data: {len(all_data)}")
print(f"NUM_CLASSES set to: {NUM_CLASSES}")


Classes detected: ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
Total Data: 21760
NUM_CLASSES set to: 7


In [11]:
# # The -v flag will show you the compilation logs so you know it's working
# !pip install mamba-ssm

  Using cached mamba_ssm-2.2.6.post3.tar.gz (113 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mamba-ssm: filename=mamba_ssm-2.2.6.post3-cp311-cp311-linux_x86_64.whl size=370549255 sha256=b848a631f8705ab626c819c036071b95cd45c69d4c513e51ea3b009c77134509
  Stored in directory: /root/.cache/pip/wheels/20/8d/74/9442d773e32cab307c5261b5e4a2c585f76f3e6fba6808e0b2
Successfully built mamba-ssm


In [13]:
!nvidia-smi

Sun Nov 30 20:10:05 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
# !pip list | grep torch

pytorch-ignite                           0.5.3
pytorch-lightning                        2.5.5
torch                                    2.6.0+cu124
torchao                                  0.10.0
torchaudio                               2.6.0+cu124
torchdata                                0.11.0
torchinfo                                1.8.0
torchmetrics                             1.8.2
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.21.0+cu124


In [19]:
# !pip3 install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 901.0/901.0 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 100.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 21.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 9.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 29.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 6.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 50.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [20]:
# !pip list | grep torch

pytorch-ignite                           0.5.3
pytorch-lightning                        2.5.5
torch                                    2.9.1+cu128
torchao                                  0.10.0
torchaudio                               2.9.1+cu128
torchdata                                0.11.0
torchinfo                                1.8.0
torchmetrics                             1.8.2
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.24.1+cu128


In [1]:
from mamba_ssm import Mamba
print("✅ Mamba loaded successfully")

✅ Mamba loaded successfully


In [11]:
import torch.optim as optim
from transformers import AutoModelForImageClassification

# 1. Load Model
print("Loading MambaVision Model...")
model = AutoModelForImageClassification.from_pretrained(
    "nvidia/MambaVision-T-1K",
    trust_remote_code=True,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)
model = model.to(device)

# -------------------------------------------------------
# PARTIAL FREEZING STRATEGY (ROBUST VERSION)
# -------------------------------------------------------

# Step A: Freeze EVERYTHING first
for param in model.parameters():
    param.requires_grad = False

# Step B: Find and Unfreeze the Classifier Head
print("--- Locating Classifier Head ---")
# Check common names for the final layer
head_found = False
possible_head_names = ['head', 'classifier', 'fc']

# 1. Check top-level attributes
for name in possible_head_names:
    if hasattr(model, name):
        print(f"Found head at: model.{name}")
        for param in getattr(model, name).parameters():
            param.requires_grad = True
        head_found = True
        break

# 2. Check one level deeper (often wrapped in 'model' or 'backbone')
if not head_found and hasattr(model, 'model'): # MambaVision specific wrapper
    for name in possible_head_names:
        if hasattr(model.model, name):
            print(f"Found head at: model.model.{name}")
            for param in getattr(model.model, name).parameters():
                param.requires_grad = True
            head_found = True
            break

if not head_found:
    print("⚠️ CRITICAL: Could not find the head automatically.")
    print("Printing last children of model to debug:")
    print(list(model.named_children())[-1]) 

# Step C: Unfreeze the Last Block / Level
print("\n--- Locating Last Stage ---")

if hasattr(model, 'levels'):
    print("Found levels at: model.levels")
    for param in model.levels[-1].parameters():
        param.requires_grad = True
elif hasattr(model, 'model') and hasattr(model.model, 'levels'):
    print("Found levels at: model.model.levels")
    for param in model.model.levels[-1].parameters():
        param.requires_grad = True
else:
    print("⚠️ CRITICAL: Could not find 'levels'.")

# -------------------------------------------------------
# VERIFICATION
# -------------------------------------------------------
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable Parameters: {trainable_params:,} / {total_params:,}")
print(f"Percentage Trainable: {100 * trainable_params / total_params:.2f}%")

# -------------------------------------------------------
# OPTIMIZER SETUP
# -------------------------------------------------------
# Using a lower learning rate (1e-4) since we are fine-tuning the last block
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)
print("Optimizer ready.")

Loading MambaVision Model...
--- Locating Classifier Head ---
Found head at: model.model.head

--- Locating Last Stage ---
Found levels at: model.model.levels

Trainable Parameters: 18,763,240 / 31,794,248
Percentage Trainable: 59.01%
Optimizer ready.


In [ ]:
# # ---------------------------------------------------------
# # MambaVision Setup
# # ---------------------------------------------------------
# print("\nInitializing MambaVision...")

# # model = AutoModelForImageClassification.from_pretrained(
# #     "nvidia/MambaVision-T-1K",
# #     trust_remote_code=True,       # Loads the custom code that imports mamba_ssm
# #     num_labels=NUM_CLASSES,       # Resizes head to 7 classes
# #     ignore_mismatched_sizes=True 
# # )

# # model = model.to(device)

# # criterion = nn.CrossEntropyLoss()
# # optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)

# # print("MambaVision initialized successfully.")


# model = AutoModelForImageClassification.from_pretrained(
#     "nvidia/MambaVision-T-1K",
#     trust_remote_code=True,
#     num_labels=NUM_CLASSES,
#     ignore_mismatched_sizes=True
# )

# # 2. Freeze all parameters first
# for param in model.parameters():
#     param.requires_grad = False

# # 3. Unfreeze ONLY the classifier head (The last layer)
# # In HuggingFace models, the head is usually named 'classifier'
# for param in model.classifier.parameters():
#     param.requires_grad = True

# model = model.to(device)
# criterion = nn.CrossEntropyLoss()

# # 4. Pass only the trainable parameters to the optimizer (Optional but cleaner)
# optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

In [16]:
import os
import time
import torch
import torch.nn as nn
from tqdm.auto import tqdm  # Import the progress bar library

# ---------------------------------------------------------
# 2. TRAINING FUNCTION (With Progress Bar)
# ---------------------------------------------------------
def train_model(model, train_loader, val_loader, epochs=10):
    train_acc, val_acc, train_loss, val_loss = [], [], [], []
    final_train_preds, final_train_labels = [], []
    final_val_preds, final_val_labels = [], []
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # --- Training Phase ---
        model.train()
        correct, total, running_loss = 0, 0, 0.0
        all_preds, all_labels = [], []
        
        # Wrap train_loader with tqdm for the progress bar
        train_bar = tqdm(train_loader, desc="Training", leave=False)
        
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            outputs = model(inputs)
            
            # ✅ FIX: Use dictionary access ['logits']
            loss = criterion(outputs['logits'], labels) 
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs['logits'], 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Update progress bar with current loss
            train_bar.set_postfix(loss=loss.item())
        
        epoch_train_acc = 100 * correct / total
        epoch_train_loss = running_loss / len(train_loader)
        final_train_preds, final_train_labels = all_preds, all_labels

        # --- Validation Phase ---
        model.eval()
        correct, total, running_loss = 0, 0, 0.0
        all_preds, all_labels = [], []
        
        # Wrap val_loader with tqdm
        val_bar = tqdm(val_loader, desc="Validating", leave=False)
        
        with torch.no_grad():
            for inputs, labels in val_bar:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                
                # ✅ FIX: Use dictionary access ['logits']
                loss = criterion(outputs['logits'], labels)
                running_loss += loss.item()
                _, predicted = torch.max(outputs['logits'], 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        epoch_val_acc = 100 * correct / total
        epoch_val_loss = running_loss / len(val_loader)
        final_val_preds, final_val_labels = all_preds, all_labels
        
        # --- Save history ---
        train_acc.append(epoch_train_acc)
        val_acc.append(epoch_val_acc)
        train_loss.append(epoch_train_loss)
        val_loss.append(epoch_val_loss)
        
        # Print Epoch Summary
        print(f"--> Train Acc: {epoch_train_acc:.2f}% | Val Acc: {epoch_val_acc:.2f}% | "
              f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

        # ✅ CHECKPOINT SAVING
        if (epoch + 1) % 5 == 0:
            checkpoint_path = f"checkpoints/mamba_whole_model_epoch_{epoch+1}.pth"
            torch.save(model, checkpoint_path) 
            print(f"--> Checkpoint saved: {checkpoint_path}")
            
    return train_acc, val_acc, train_loss, val_loss, final_train_preds, final_train_labels, final_val_preds, final_val_labels

# ---------------------------------------------------------
# 3. EXECUTION (With Timers & Metrics)
# ---------------------------------------------------------

# A. Stats before training
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

# B. Start Training
EPOCHS = 25
print(f"\nStarting training for {EPOCHS} epochs...")
start_train = time.time()

# Run the training loop
train_acc, val_acc, train_loss, val_loss, train_preds, train_labels, val_preds, val_labels = train_model(model, train_loader, val_loader, EPOCHS)

end_train = time.time()
training_time_minutes = (end_train - start_train) / 60
print(f"\nTraining finished in {training_time_minutes:.2f} minutes.")

# C. Save Final Model
final_model_path = "/kaggle/working/mamba_emotion_whole_final.pth"
torch.save(model, final_model_path)
print(f"Final whole model saved at: {final_model_path}")


Starting training for 25 epochs...

Epoch 1/25


Training:   0%|          | 0/544 [00:00<?, ?it/s]

Validating:   0%|          | 0/68 [00:00<?, ?it/s]

--> Train Acc: 56.89% | Val Acc: 68.34% | Train Loss: 1.5634 | Val Loss: 0.9039

Epoch 2/25


Training:   0%|          | 0/544 [00:00<?, ?it/s]

Validating:   0%|          | 0/68 [00:00<?, ?it/s]

... (66 outputs left out)

--> Train Acc: 83.79% | Val Acc: 81.71% | Train Loss: 0.4330 | Val Loss: 0.5119

Epoch 25/25


Training:   0%|          | 0/544 [00:00<?, ?it/s]

Validating:   0%|          | 0/68 [00:00<?, ?it/s]

--> Train Acc: 84.29% | Val Acc: 81.48% | Train Loss: 0.4289 | Val Loss: 0.5222
--> Checkpoint saved: checkpoints/mamba_whole_model_epoch_25.pth

Training finished in 129.89 minutes.
Final whole model saved at: /kaggle/working/mamba_emotion_whole_final.pth


In [ ]:
import torch
import torch.optim as optim
import time
import pandas as pd
from tqdm.auto import tqdm

# ---------------------------------------------------------
# 1. Configuration & Setup
# ---------------------------------------------------------
PREV_EPOCHS = 25
EXTRA_EPOCHS = 10 
CONTINUE_PATH = "/kaggle/working/mamba_emotion_whole_final.pth"

# Load Model
print(f"⏳ Loading model from Epoch {PREV_EPOCHS}...")
model = torch.load(CONTINUE_PATH, weights_only=False)
model = model.to(device)

# # Re-initialize Optimizer (Keeping same LR=1e-4 as requested)
criterion = nn.CrossEntropyLoss()
# optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-4)

# ---------------------------------------------------------
# 2. Define Function (Appends DIRECTLY to global lists)
# ---------------------------------------------------------
def train_continue_direct(model, train_loader, val_loader, epochs, start_epoch):
    
    # We use the GLOBAL lists directly
    # This modifies the lists that exist outside the function
    global train_acc, val_acc, train_loss, val_loss
    
    # Vars for confusion matrix (we only need the last epoch's preds)
    final_preds, final_labels = [], []
    
    end_epoch = start_epoch + epochs
    
    for epoch in range(start_epoch, end_epoch):
        print(f"\nEpoch {epoch+1}/{end_epoch}")
        
        # --- Training ---
        model.train()
        correct, total, running_loss = 0, 0, 0.0
        
        train_bar = tqdm(train_loader, desc=f"Training {epoch+1}", leave=False)
        
        for inputs, labels in train_bar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs['logits'], labels) 
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs['logits'], 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            train_bar.set_postfix(loss=loss.item())
        
        # Calculate Epoch Stats
        epoch_t_acc = 100 * correct / total
        epoch_t_loss = running_loss / len(train_loader)

        # --- Validation ---
        model.eval()
        correct, total, running_loss = 0, 0, 0.0
        all_preds, all_labels = [], []
        
        val_bar = tqdm(val_loader, desc="Validating", leave=False)
        
        with torch.no_grad():
            for inputs, labels in val_bar:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs['logits'], labels)
                running_loss += loss.item()
                _, predicted = torch.max(outputs['logits'], 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                # Save predictions for the very last epoch
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        epoch_v_acc = 100 * correct / total
        epoch_v_loss = running_loss / len(val_loader)
        
        # --- DIRECTLY APPEND TO GLOBAL LISTS ---
        train_acc.append(epoch_t_acc)
        val_acc.append(epoch_v_acc)
        train_loss.append(epoch_t_loss)
        val_loss.append(epoch_v_loss)
        
        print(f"--> Train Acc: {epoch_t_acc:.2f}% | Val Acc: {epoch_v_acc:.2f}% | "
              f"Train Loss: {epoch_t_loss:.4f} | Val Loss: {epoch_v_loss:.4f}")

        # Checkpoint
        if (epoch + 1) % 2 == 0:
            torch.save(model, f"/kaggle/working/checkpoints/mamba_whole_model_epoch_{epoch+1}.pth")
    
    return all_preds, all_labels

# ---------------------------------------------------------
# 3. Execution
# ---------------------------------------------------------

# Ensure lists exist (Safety check if kernel restarted)
if 'train_acc' not in globals():
    print("⚠️ Warning: Previous history not found. Creating new empty lists.")
    train_acc, val_acc, train_loss, val_loss = [], [], [], []

print(f"\n🚀 Resuming training from Epoch {PREV_EPOCHS + 1} to {PREV_EPOCHS + EXTRA_EPOCHS}...")
start_time = time.time()

# Run training (updates lists in-place)
test_preds, test_labels = train_continue_direct(
    model, 
    train_loader, 
    val_loader, 
    epochs=EXTRA_EPOCHS, 
    start_epoch=PREV_EPOCHS
)

print(f"\nContinued training finished in {(time.time() - start_time)/60:.2f} minutes.")

# ---------------------------------------------------------
# 4. Save CSV and Model
# ---------------------------------------------------------
# Save updated history
history_data = {
    'Epoch': range(1, len(train_acc) + 1),
    'Train_Accuracy': train_acc,
    'Val_Accuracy': val_acc,
    'Train_Loss': train_loss,
    'Val_Loss': val_loss
}

df_full_history = pd.DataFrame(history_data)
csv_filename = "training_history_full.csv"
df_full_history.to_csv(csv_filename, index=False)
print(f"✅ Full metrics saved to '{csv_filename}' (Total Epochs: {len(train_acc)})")

# Save Final Model
final_extended_path = "/kaggle/working/mamba_emotion_extended_final_>25epoch.pth"
torch.save(model, final_extended_path)
print(f"✅ Extended model saved to: {final_extended_path}")

⏳ Loading model from Epoch 25...

🚀 Resuming training from Epoch 26 to 35...

Epoch 26/35


Training 26:   0%|          | 0/544 [00:00<?, ?it/s]

Validating:   0%|          | 0/68 [00:00<?, ?it/s]

--> Train Acc: 84.29% | Val Acc: 81.43% | Train Loss: 0.4216 | Val Loss: 0.5176

Epoch 27/35


Training 27:   0%|          | 0/544 [00:00<?, ?it/s]

Validating:   0%|          | 0/68 [00:00<?, ?it/s]

... (19 outputs left out)

Training 34:   0%|          | 0/544 [00:00<?, ?it/s]

Validating:   0%|          | 0/68 [00:00<?, ?it/s]

--> Train Acc: 84.73% | Val Acc: 81.43% | Train Loss: 0.4184 | Val Loss: 0.5125

Epoch 35/35


Training 35:   0%|          | 0/544 [00:00<?, ?it/s]

In [43]:
print('hi')

hi


In [41]:
import pandas as pd
import os

# 1. Check if the lists exist and have data
if 'train_acc' in globals() and len(train_acc) > 0:
    
    # 2. Create a dictionary of your data
    history_data = {
        'Epoch': range(1, len(train_acc) + 1),
        'Train_Accuracy': train_acc,
        'Val_Accuracy': val_acc,
        'Train_Loss': train_loss,
        'Val_Loss': val_loss
    }
    
    # 3. Convert to DataFrame
    df_history = pd.DataFrame(history_data)
    
    # 4. Save to CSV
    filename = "/kaggle/working/training_history_epoch0-25.csv"
    df_history.to_csv(filename, index=False)
    
    print(f"✅ Success! Saved metrics for {len(train_acc)} epochs to '{filename}'.")
    print("Preview of the data:")
    print(df_history.tail()) # Show last 5 rows

else:
    print("⚠️ Error: No training history found in memory. Did you run the first training loop?")

✅ Success! Saved metrics for 25 epochs to '/kaggle/working/training_history_epoch0-25.csv'.
Preview of the data:
    Epoch  Train_Accuracy  Val_Accuracy  Train_Loss  Val_Loss
20     21       82.628676     82.306985    0.469496  0.468627
21     22       83.231847     82.215074    0.456208  0.467646
22     23       83.547794     80.330882    0.445709  0.511436
23     24       83.794807     81.709559    0.432965  0.511939
24     25       84.294577     81.479779    0.428911  0.522171


In [44]:
import pandas as pd
import os

# 1. Check if the lists exist and have data
if 'train_acc' in globals() and len(train_acc) > 0:
    
    # 2. Create a dictionary of your data
    history_data = {
        'Epoch': range(1, len(train_acc) + 1),
        'Train_Accuracy': train_acc,
        'Val_Accuracy': val_acc,
        'Train_Loss': train_loss,
        'Val_Loss': val_loss
    }
    
    # 3. Convert to DataFrame
    df_history = pd.DataFrame(history_data)
    
    # 4. Save to CSV
    filename = "/kaggle/working/training_history_epoch0-34.csv"
    df_history.to_csv(filename, index=False)
    
    print(f"✅ Success! Saved metrics for {len(train_acc)} epochs to '{filename}'.")
    print("Preview of the data:")
    print(df_history.tail()) # Show last 5 rows

else:
    print("⚠️ Error: No training history found in memory. Did you run the first training loop?")

✅ Success! Saved metrics for 35 epochs to '/kaggle/working/training_history_epoch0-34.csv'.
Preview of the data:
    Epoch  Train_Accuracy  Val_Accuracy  Train_Loss  Val_Loss
30     31       84.587546     81.893382    0.417012  0.513561
31     32       84.725414     81.295956    0.412698  0.537783
32     33       84.518612     81.571691    0.418374  0.524898
33     34       84.731158     81.433824    0.418360  0.512473
34     35       84.363511     81.387868    0.421634  0.524242


In [34]:
# ---------------------------------------------------------
# 7. Evaluation & Metrics
# ---------------------------------------------------------
def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_labels = [], []
    correct, total = 0, 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            
            # ✅ FIX: Use dictionary access ['logits']
            _, predicted = torch.max(outputs['logits'], 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_acc = 100 * correct / total
    test_error = 100 - test_acc

    print(f"\nTest Accuracy: {test_acc:.2f}% | Test Error: {test_error:.2f}%")
    print("\nTest Classification Report:")
    # Ensure full_dataset.classes or similar variable is available
    # If using the variable 'class_names' we defined earlier:
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    
    return all_preds, all_labels, test_acc, test_error

In [22]:
test_preds, test_labels, test_acc, test_error = evaluate_model(model, test_loader)


Test Accuracy: 81.39% | Test Error: 18.61%

Test Classification Report:
              precision    recall  f1-score   support

       Angry     0.7123    0.8125    0.7591       128
     Disgust     0.8612    0.9930    0.9225       575
        Fear     0.9355    0.5210    0.6692       167
       Happy     0.9093    0.9237    0.9164       380
     Neutral     0.6966    0.7416    0.7184       418
         Sad     0.7774    0.6544    0.7106       379
    Surprise     0.8065    0.7752    0.7905       129

    accuracy                         0.8139      2176
   macro avg     0.8141    0.7745    0.7838      2176
weighted avg     0.8171    0.8139    0.8085      2176



In [45]:
print('epoch 0-34')
test_preds, test_labels, test_acc, test_error = evaluate_model(model, test_loader)



Test Accuracy: 81.71% | Test Error: 18.29%

Test Classification Report:
              precision    recall  f1-score   support

       Angry     0.7444    0.7734    0.7586       128
     Disgust     0.8678    0.9930    0.9262       575
        Fear     0.9375    0.5389    0.6844       167
       Happy     0.9093    0.9237    0.9164       380
     Neutral     0.6809    0.8014    0.7363       418
         Sad     0.7926    0.6253    0.6991       379
    Surprise     0.8482    0.7364    0.7884       129

    accuracy                         0.8171      2176
   macro avg     0.8258    0.7703    0.7871      2176
weighted avg     0.8230    0.8171    0.8119      2176



In [23]:
# Results DataFrame
results = {
    "Dataset": ["Training", "Validation", "Testing"],
    "Epochs": [EPOCHS, EPOCHS, EPOCHS],
    "Accuracy (%)": [train_acc[-1], val_acc[-1], test_acc],
    "Error (%)": [100 - train_acc[-1], 100 - val_acc[-1], test_error],
    "F1 Score": [
        f1_score(train_labels, train_preds, average="macro"),
        f1_score(val_labels, val_preds, average="macro"),
        f1_score(test_labels, test_preds, average="macro")
    ],
    "Recall": [
        recall_score(train_labels, train_preds, average="macro"),
        recall_score(val_labels, val_preds, average="macro"),
        recall_score(test_labels, test_preds, average="macro")
    ]
}

df_results = pd.DataFrame(results)
print("\n=== Final Performance Table ===")
print(df_results.to_string(index=False))


=== Final Performance Table ===
   Dataset  Epochs  Accuracy (%)  Error (%)  F1 Score   Recall
  Training      25     84.294577  15.705423  0.820681 0.812145
Validation      25     81.479779  18.520221  0.769869 0.755983
   Testing      25     81.387868  18.612132  0.783828 0.774480


In [47]:
# Results DataFrame
results = {
    "Dataset": ["Training", "Validation", "Testing"],
    "Epochs": [EPOCHS+9, EPOCHS+9, EPOCHS+9],
    "Accuracy (%)": [train_acc[-1], val_acc[-1], test_acc],
    "Error (%)": [100 - train_acc[-1], 100 - val_acc[-1], test_error],
    "F1 Score": [
        f1_score(train_labels, train_preds, average="macro"),
        f1_score(val_labels, val_preds, average="macro"),
        f1_score(test_labels, test_preds, average="macro")
    ],
    "Recall": [
        recall_score(train_labels, train_preds, average="macro"),
        recall_score(val_labels, val_preds, average="macro"),
        recall_score(test_labels, test_preds, average="macro")
    ]
}

df_results = pd.DataFrame(results)
print("\n=== Final Performance Table ===")
print(df_results.to_string(index=False))


=== Final Performance Table ===
   Dataset  Epochs  Accuracy (%)  Error (%)  F1 Score   Recall
  Training      34     84.363511  15.636489  0.820681 0.812145
Validation      34     81.387868  18.612132  0.769869 0.755983
   Testing      34     81.709559  18.290441  0.787062 0.770327


In [31]:

# ---------------------------------------------------------
# 8. Plotting
# ---------------------------------------------------------
epochs_run = len(val_acc)
plt.style.use('seaborn-v0_8-white')

# Acc
plt.figure(figsize=(7, 5))
plt.plot(range(1, epochs_run + 1), train_acc[:epochs_run], label='Train Acc', color='royalblue')
plt.plot(range(1, epochs_run + 1), val_acc[:epochs_run], label='Val Acc', color='darkorange')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.title('MambaVision Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

# Loss
plt.figure(figsize=(7, 5))
plt.plot(range(1, epochs_run + 1), train_loss[:epochs_run], label='Train Loss', color='royalblue')
plt.plot(range(1, epochs_run + 1), val_loss[:epochs_run], label='Val Loss', color='darkorange')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('MambaVision Loss')
plt.legend()
plt.tight_layout()
plt.show()

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=16)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# ROC Curve
model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        # KEY CHANGE: Access .logits then apply softmax
        probs = torch.softmax(outputs.logits, dim=1) 
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

y_score = np.array(all_probs)
y_true = np.array(all_labels)

# Binarize labels
y_true_binarized = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

fpr, tpr, roc_auc = {}, {}, {}
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_true_binarized.ravel(), y_score.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(10, 8))
colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'green', 'red', 'purple', 'brown'])

for i, color in zip(range(NUM_CLASSES), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f"{class_names[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot(fpr["micro"], tpr["micro"], color='deeppink', linestyle=':', linewidth=4,
         label=f"Micro-average (AUC = {roc_auc['micro']:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC (MambaVision)')
plt.legend(loc="lower right")
plt.show()

<Figure size 700x500 with 1 Axes>

<Figure size 700x500 with 1 Axes>

<Figure size 1000x800 with 2 Axes>

AttributeError: 'dict' object has no attribute 'logits'

In [48]:

# ---------------------------------------------------------
# 8. Plotting
# ---------------------------------------------------------
epochs_run = len(val_acc)
plt.style.use('seaborn-v0_8-white')

# Acc
plt.figure(figsize=(7, 5))
plt.plot(range(1, epochs_run + 1), train_acc[:epochs_run], label='Train Acc', color='royalblue')
plt.plot(range(1, epochs_run + 1), val_acc[:epochs_run], label='Val Acc', color='darkorange')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.title('MambaVision Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

# Loss
plt.figure(figsize=(7, 5))
plt.plot(range(1, epochs_run + 1), train_loss[:epochs_run], label='Train Loss', color='royalblue')
plt.plot(range(1, epochs_run + 1), val_loss[:epochs_run], label='Val Loss', color='darkorange')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('MambaVision Loss')
plt.legend()
plt.tight_layout()
plt.show()

# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=16)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# ROC Curve
model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        # KEY CHANGE: Access .logits then apply softmax
        probs = torch.softmax(outputs.logits, dim=1) 
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

y_score = np.array(all_probs)
y_true = np.array(all_labels)

# Binarize labels
y_true_binarized = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

fpr, tpr, roc_auc = {}, {}, {}
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_true_binarized.ravel(), y_score.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

plt.figure(figsize=(10, 8))
colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'green', 'red', 'purple', 'brown'])

for i, color in zip(range(NUM_CLASSES), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f"{class_names[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot(fpr["micro"], tpr["micro"], color='deeppink', linestyle=':', linewidth=4,
         label=f"Micro-average (AUC = {roc_auc['micro']:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC (MambaVision)')
plt.legend(loc="lower right")
plt.show()

<Figure size 700x500 with 1 Axes>

<Figure size 700x500 with 1 Axes>

<Figure size 1000x800 with 2 Axes>

AttributeError: 'dict' object has no attribute 'logits'

In [28]:
with torch.no_grad():
    for x, _ in test_loader:
        o = model(x.to(device))
        print("OUTPUT TYPE:", type(o))
        print("DICT KEYS:", o.keys())
        break


OUTPUT TYPE: <class 'dict'>
DICT KEYS: dict_keys(['logits'])


In [51]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------
# Change this to the path of the image you want to test
IMAGE_PATH = "/kaggle/input/custom-input/sad_face.webp" 
MODEL_PATH = "/kaggle/working/mamba_emotion_whole_final.pth"

EMOTION_LABELS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------
# 2. Load the Model
# ---------------------------------------------------------
print(f"⏳ Loading model from {MODEL_PATH}...")
# weights_only=False is required for MambaVision
model = torch.load(MODEL_PATH, weights_only=False)
model.to(device)
model.eval()
print("✅ Model loaded.")

# ---------------------------------------------------------
# 3. Preprocessing (Must match training!)
# ---------------------------------------------------------
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Standard ImageNet normalization
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ---------------------------------------------------------
# 4. Run Inference
# ---------------------------------------------------------
def predict_image(image_path):
    # Open image
    try:
        image = Image.open(image_path).convert('RGB')
    except Exception as e:
        print(f"Error opening image: {e}")
        return

    # Preprocess
    input_tensor = preprocess(image)
    
    # Add batch dimension [3, 224, 224] -> [1, 3, 224, 224]
    input_batch = input_tensor.unsqueeze(0).to(device)

    # Inference
    with torch.no_grad():
        outputs = model(input_batch)
        
        # Handle MambaVision dictionary output
        if isinstance(outputs, dict):
            logits = outputs['logits']
        else:
            logits = outputs.logits
            
        # Get probabilities
        probs = F.softmax(logits, dim=1)
        confidence, predicted_idx = torch.max(probs, 1)

    # Get Result info
    predicted_label = EMOTION_LABELS[predicted_idx.item()]
    confidence_score = confidence.item() * 100

    # ---------------------------------------------------------
    # 5. Visualize Result
    # ---------------------------------------------------------
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis('off')
    
    # Set title color based on confidence
    title_color = 'green' if confidence_score > 70 else 'red'
    plt.title(f"Prediction: {predicted_label}\nConfidence: {confidence_score:.2f}%", 
              color=title_color, fontsize=14, fontweight='bold')
    plt.show()

    # Print top 3 probabilities for analysis
    print(f"\n--- Top 3 Predictions ---")
    top3_prob, top3_idx = torch.topk(probs, 3)
    for i in range(3):
        label = EMOTION_LABELS[top3_idx[0][i].item()]
        score = top3_prob[0][i].item() * 100
        print(f"{i+1}. {label}: {score:.2f}%")

# Run the function
predict_image(IMAGE_PATH)

⏳ Loading model from /kaggle/working/mamba_emotion_whole_final.pth...
✅ Model loaded.


<Figure size 600x600 with 1 Axes>


--- Top 3 Predictions ---
1. Sad: 92.92%
2. Neutral: 6.76%
3. Happy: 0.15%


In [3]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoModelForImageClassification

# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------
IMAGE_PATH = "/kaggle/input/team-member-face-expression/WhatsApp Image 2025-12-01 at 08.49.39.jpeg" 
MODEL_PATH = "/kaggle/working/mamba_emotion_extended_final_>25epoch.pth"

EMOTION_LABELS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------
# 2. Re-Download Model Definition (REQUIRED)
# ---------------------------------------------------------
# Even with persistence, the specific temporary folder for the MambaVision 
# class definition is lost on restart. We must download it again.
print("⏳ restoring model class definition...")
try:
    # This recreates the missing 'transformers_modules' folder
    dummy = AutoModelForImageClassification.from_pretrained(
        "nvidia/MambaVision-T-1K", 
        trust_remote_code=True
    )
    print("✅ System paths restored.")
except Exception as e:
    print(f"⚠️ Warning: Could not download base model. If inference fails, check internet. Error: {e}")

# ---------------------------------------------------------
# 3. Load Your Trained Model
# ---------------------------------------------------------
print(f"⏳ Loading your trained weights from {MODEL_PATH}...")
model = torch.load(MODEL_PATH, weights_only=False)
model.to(device)
model.eval()
print("✅ Custom Model loaded successfully.")

# ---------------------------------------------------------
# 4. Preprocessing
# ---------------------------------------------------------
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ---------------------------------------------------------
# 5. Inference Function
# ---------------------------------------------------------
def predict_image(image_path):
    try:
        image = Image.open(image_path).convert('RGB')
    except Exception as e:
        print(f"❌ Error opening image: {e}")
        return

    input_tensor = preprocess(image)
    input_batch = input_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_batch)
        
        # Handle different output types
        if isinstance(outputs, dict):
            logits = outputs['logits']
        else:
            logits = outputs.logits
            
        probs = F.softmax(logits, dim=1)
        confidence, predicted_idx = torch.max(probs, 1)

    predicted_label = EMOTION_LABELS[predicted_idx.item()]
    confidence_score = confidence.item() * 100

    # Visualization
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis('off')
    
    title_color = 'green' if confidence_score > 70 else 'red'
    plt.title(f"Prediction: {predicted_label}\nConfidence: {confidence_score:.2f}%", 
              color=title_color, fontsize=14, fontweight='bold')
    plt.show()

    print(f"\n--- Top 3 Predictions ---")
    top3_prob, top3_idx = torch.topk(probs, 3)
    for i in range(3):
        label = EMOTION_LABELS[top3_idx[0][i].item()]
        score = top3_prob[0][i].item() * 100
        print(f"{i+1}. {label}: {score:.2f}%")

# Run
predict_image(IMAGE_PATH)

⏳ restoring model class definition...


config.json: 0.00B [00:00, ?B/s]

configuration_mambavision.py:   0%|          | 0.00/625 [00:00<?, ?B/s]

modeling_mambavision.py: 0.00B [00:00, ?B/s]

⚠️ Warning: Could not download base model. If inference fails, check internet. Error: This modeling file requires the following packages that were not found in your environment: mamba_ssm. Run `pip install mamba_ssm`
⏳ Loading your trained weights from /kaggle/working/mamba_emotion_extended_final_>25epoch.pth...


ModuleNotFoundError: No module named 'transformers_modules.nvidia.MambaVision-T-1K.b1de77e17599566d98efb701c0231b1095dc3a67.modeling_mambavision'

In [4]:
from mamba_ssm import Mamba
print("✅ Mamba loaded successfully")

ModuleNotFoundError: No module named 'mamba_ssm'